# Get data from TOP500+
- Article (in Latvian): https://aivis.medium.com/top-500-lielākie-uzņēmumi-latvijā-saraksta-izgūšana-ar-llm-api-7820a67ccdac

In [3]:
import os
import io
import base64
import csv
import glob
import anthropic # pip install anthropic
from dotenv import load_dotenv # pip install python-dotenv
from google import genai # pip install google-genai
from google.genai import types # pip install google-genai
from mistralai import Mistral # pip install mistralai
from openai import OpenAI # pip install openai
import pandas as pd # pip install pandas
from PIL import Image # pip install pillow

In [4]:
# Load the .env file
load_dotenv()

True

In [ ]:
PREP_CLAUDE = True
PREP_OPENAI = True
PREP_GEMINI = True

In [ ]:
# Location of the project
BASE_DIR = os.path.dirname('TOP500_OCR')

# Sub-directory - where images are located
IMAGE_DIR = os.path.join(BASE_DIR, 'pics')

In [ ]:
claude_api_key = os.environ.get("ANTHROPIC_API_KEY") if PREP_CLAUDE else None
openai_api_key = os.environ.get("OPENAI_API_KEY") if PREP_OPENAI else None
gemini_api_key = os.environ.get("GEMINI_API_KEY") if PREP_GEMINI else None

In [ ]:
# Add/Remove/Select models that you need to use
models = {# https://platform.claude.com/docs/en/about-claude/models/overview
    "claude": ["claude-opus-4-5-20251101",      # Input: 5.00 USD; output: 25.00 USD
               "claude-sonnet-4-5-20250929",    # Input: 3.00 USD; output: 15.00 USD
               "claude-haiku-4-5-20251001"      # Input: 1.00 USD; output:  5.00 USD
               ],    
    # https://platform.openai.com/docs/models
    "openai": ["gpt-5.2-2025-12-11",            # Input: 1.75 USD; output: 14.00 USD
               #"gpt-5.1-2025-11-13",            # Input: 1.25 USD; output: 10.00 USD
               #"gpt-5-mini-2025-08-07",         # Input: 0.25 USD; output:  2.00 USD
               #"gpt-5-nano-2025-08-07"          # Input: 0.05 USD; output:  0.40 USD
               ],        
    # https://ai.google.dev/gemini-api/docs/models and https://ai.google.dev/gemini-api/docs/pricing
    "google": ["gemini-3-pro-preview",
               "gemini-2.5-flash",
               #"gemini-2.5-flash-lite"
               ],
    # https://docs.mistral.ai/getting-started/models
    # "mistral": ["mistral-large-2512",           # Input: 0.50 USD; output:  1.50 USD
    #             "mistral-medium-2508",          # Input: 0.40 USD; output:  2.00 USD
    #             "mistral-ocr-2505"]             # Input: 1.00 USD; output:  3.00 USD (per 1000 pages)
          }

In [9]:
def get_images(directory):
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp']
    images = []
    for ext in image_extensions:
        # collecting both lowercase and we could add uppercase validation if needed, 
        # but glob on windows is often case insensitive. 
        # For robustness we can just look for the list.
        images.extend(glob.glob(os.path.join(directory, ext)))
        # check uppercase extensions too just in case (Linux/strict envs)
        images.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(list(set(images))) # Remove duplicates if filesystem is case-insensitive


In [22]:
prompt = """Here is a table from magazine in Latvian language.
Read data from this image and give me the result in csv format (separate data by pipe: | ) with following columns:
- rank_24 (rank position in 2024),
- rank_23 (rank position in 2023),
- company (company name in Latvian),
- industry (industry in Latvian),
- turnover_m_eur,
- changes_23,
- profit_m_eur.
Return only data, do not add additional comments"""

## Anthropic

In [31]:
if PREP_CLAUDE:
    client = anthropic.Anthropic(api_key=claude_api_key)


In [ ]:
def process_images(model):
    if not os.path.exists(IMAGE_DIR):
        print(f"Directory not found: {IMAGE_DIR}")
        return

    list_images = get_images(IMAGE_DIR)
    
    if not list_images:
        print(f"No images found in {IMAGE_DIR}")
        return

    print(f"Found {len(list_images)} images to process.")

    for m in models[model]:
        print(f'Model: {m}')

        # Open the single output file in write mode
        OUTPUT_FILE = f"TOP500_2025_{m}.csv"
        with open(os.path.join(BASE_DIR, OUTPUT_FILE), mode="w", encoding="utf-8", newline="") as csvfile:
            writer = csv.writer(csvfile)

            for i in list_images:
                try:
                    print(f"Processing: {os.path.basename(i)}")
                    with Image.open(i) as img:
                        # Resize the image to 951x1268
                        # using LANCZOS for high quality downsampling if needed, though default is usually fine and user example used default.
                        resized_img = img.resize((951, 1268)) # https://platform.claude.com/docs/en/build-with-claude/vision

                        # Save the resized image to an in-memory buffer
                        buffer = io.BytesIO()
                        resized_img.save(buffer, format="JPEG")
                        buffer.seek(0) # Reset pointer to the start

                        # Encode the buffer to Base64
                        base64_encoded_image = base64.b64encode(buffer.read()).decode("utf-8")

                    message = client.messages.create(
                        model=m,
                        max_tokens=8192,
                        temperature=0,
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": "image/jpeg",
                                            "data": base64_encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ],
                    )
                
                    print(f"Image: {os.path.basename(i)}; Input tokens: {message.usage.input_tokens}; Output tokens: {message.usage.output_tokens}")

                    # Split the string into lines
                    # Handle potential empty lines or preambles if the model is chatty (though prompt says return only data)
                    content = message.content[0].text.strip()
                    lines = content.split("\n")

                    for line in lines:
                        if line.strip():
                            row = line.split("|")
                            # Prepend the image filename
                            row.insert(0, os.path.basename(i))
                            writer.writerow(row)
                    
                    # Flush to ensure data is written incrementally in case of crash (optional but good practice for long jobs)
                    csvfile.flush()
                except Exception as e:
                    print(f"Error processing {os.path.basename(i)}: {str(e)}")
        
        print("-" * 30)
        print(f"Processing complete. All data saved to {OUTPUT_FILE}")

Run the code for Anthropic models (*.csv):

In [ ]:
process_images('claude')

## Google

In [ ]:
if PREP_GEMINI:
    client = genai.Client(api_key=gemini_api_key)

Result in the *.csv:

In [ ]:
def process_images(model):
    if not os.path.exists(IMAGE_DIR):
        print(f"Directory not found: {IMAGE_DIR}")
        return

    list_images = get_images(IMAGE_DIR)
    
    if not list_images:
        print(f"No images found in {IMAGE_DIR}")
        return

    print(f"Found {len(list_images)} images to process.")

    for m in models[model]:
        print(f'Model: {m}')

        # Open the single output file in write mode
        OUTPUT_FILE = f"TOP500_2025_{m}.csv"
        with open(os.path.join(BASE_DIR, OUTPUT_FILE), mode="w", encoding="utf-8", newline="") as csvfile:
            writer = csv.writer(csvfile)

            for i in list_images:
                try:
                    print(f"Processing: {os.path.basename(i)}")
                    with Image.open(i) as img:
                        # Resize the image to 951x1268
                        # using LANCZOS for high quality downsampling if needed, though default is usually fine and user example used default.
                        resized_img = img.resize((951, 1268))

                        # Save the resized image to an in-memory buffer
                        buffer = io.BytesIO()
                        resized_img.save(buffer, format="JPEG")
                        buffer.seek(0) # Reset pointer to the start

                        # Encode the buffer to Base64
                        base64_encoded_image = base64.b64encode(buffer.read()).decode("utf-8")

                    message = client.models.generate_content(
                        model=m,
                        contents=[
                            types.Part.from_bytes(
                                data=base64_encoded_image,
                                mime_type='image/jpeg',
                                ),
                                prompt
                                ]
                                )
                
                    print(f"Image: {os.path.basename(i)}; Input tokens: {message.usage_metadata.prompt_token_count}; Output tokens: {message.usage_metadata.candidates_token_count}")

                    # Split the string into lines
                    # Handle potential empty lines or preambles if the model is chatty (though prompt says return only data)
                    content = message.text.strip()
                    lines = content.split("\n")

                    for line in lines:
                        if line.strip():
                            row = line.split("|")
                            # Prepend the image filename
                            row.insert(0, os.path.basename(i))
                            writer.writerow(row)
                    
                    # Flush to ensure data is written incrementally in case of crash (optional but good practice for long jobs)
                    csvfile.flush()
                except Exception as e:
                    print(f"Error processing {os.path.basename(i)}: {str(e)}")
        
        print("-" * 30)
        print(f"Processing complete. All data saved to {OUTPUT_FILE}")

Run the code for Google models (*.csv):

In [ ]:
process_images('google')

Result in the Excel format:

In [23]:
def process_images(model):
    if not os.path.exists(IMAGE_DIR):
        print(f"Directory not found: {IMAGE_DIR}")
        return

    list_images = get_images(IMAGE_DIR)
    
    if not list_images:
        print(f"No images found in {IMAGE_DIR}")
        return

    print(f"Found {len(list_images)} images to process.")

    for m in models[model]:
        print(f'Model: {m}')

        # Prepare output file name (Excel format)
        OUTPUT_FILE = f"TOP500_2025_{m}.xlsx"
        
        # List to collect all rows
        all_rows = []

        for i in list_images:
            try:
                print(f"Processing: {os.path.basename(i)}")
                with Image.open(i) as img:
                    # Resize the image to 951x1268
                    resized_img = img.resize((951, 1268))

                    # Save the resized image to an in-memory buffer
                    buffer = io.BytesIO()
                    resized_img.save(buffer, format="JPEG")
                    buffer.seek(0)

                    # Encode the buffer to Base64
                    base64_encoded_image = base64.b64encode(buffer.read()).decode("utf-8")

                message = client.models.generate_content(
                    model=m,
                    contents=[
                        types.Part.from_bytes(
                            data=base64_encoded_image,
                            mime_type='image/jpeg',
                        ),
                        prompt
                    ]
                )
            
                print(f"Image: {os.path.basename(i)}; Input tokens: {message.usage_metadata.prompt_token_count}; Output tokens: {message.usage_metadata.candidates_token_count}")

                # Split the string into lines
                content = message.text.strip()
                lines = content.split("\n")

                for line in lines:
                    if line.strip():
                        row = line.split("|")
                        # Prepend the image filename
                        row.insert(0, os.path.basename(i))
                        all_rows.append(row)
                
            except Exception as e:
                print(f"Error processing {os.path.basename(i)}: {str(e)}")
        
        # Save all collected rows to Excel
        if all_rows:
            df = pd.DataFrame(all_rows)
            output_path = os.path.join(BASE_DIR, OUTPUT_FILE)
            df.to_excel(output_path, index=False, header=False, engine='openpyxl')
            print(f"Processing complete. All data saved to {OUTPUT_FILE}")
        else:
            print("No data to save.")
        
        print("-" * 30)

Run the code for Google models (*.xlsx):

In [ ]:
process_images('google')

## OpenAI

In [ ]:
if PREP_OPENAI:
    client = OpenAI(api_key = openai_api_key)

In [ ]:
def process_images(model):
    if not os.path.exists(IMAGE_DIR):
        print(f"Directory not found: {IMAGE_DIR}")
        return

    list_images = get_images(IMAGE_DIR)
    
    if not list_images:
        print(f"No images found in {IMAGE_DIR}")
        return

    print(f"Found {len(list_images)} images to process.")

    for m in models[model]:
        print(f'Model: {m}')

        # gpt-5-mini and gpt-5-nano doesn't support temperature=0
        TEMPERATURE = 0 if m == 'gpt-5.2-2025-12-11' else 1

        # Open the single output file in write mode
        OUTPUT_FILE = f"TOP500_2025_{m}.csv"
        with open(os.path.join(BASE_DIR, OUTPUT_FILE), mode="w", encoding="utf-8", newline="") as csvfile:
            writer = csv.writer(csvfile)

            for i in list_images:
                try:
                    print(f"Processing: {os.path.basename(i)}")
                    with Image.open(i) as img:
                        # Resize the image to 951x1268
                        # using LANCZOS for high quality downsampling if needed, though default is usually fine and user example used default.
                        resized_img = img.resize((951, 1268)) # 

                        # Save the resized image to an in-memory buffer
                        buffer = io.BytesIO()
                        resized_img.save(buffer, format="JPEG")
                        buffer.seek(0) # Reset pointer to the start

                        # Encode the buffer to Base64
                        base64_encoded_image = base64.b64encode(buffer.read()).decode("utf-8")

                    message = client.responses.create(
                        model=m,
                        max_output_tokens=8192,
                        temperature=TEMPERATURE,
                        input=[
                            {
                                "role": "user",
                                "content": [
                                    {"type": "input_text",
                                     "text": prompt },
                                     {"type": "input_image",
                                      "image_url": f"data:image/jpeg;base64,{base64_encoded_image}" }
                                            ]
                            }]
                                                        )
                
                    print(f"Image: {os.path.basename(i)}; Input tokens: {message.usage.input_tokens}; Output tokens: {message.usage.output_tokens}")

                    # Split the string into lines
                    # Handle potential empty lines or preambles if the model is chatty (though prompt says return only data)
                    content = message.output_text.strip()
                    lines = content.split("\n")

                    for line in lines:
                        if line.strip():
                            row = line.split("|")
                            # Prepend the image filename
                            row.insert(0, os.path.basename(i))
                            writer.writerow(row)
                    
                    # Flush to ensure data is written incrementally in case of crash (optional but good practice for long jobs)
                    csvfile.flush()
                except Exception as e:
                    print(f"Error processing {os.path.basename(i)}: {str(e)}")
        
        print("-" * 30)
        print(f"Processing complete. All data saved to {OUTPUT_FILE}")

Run the code for OpenAI models (*.csv):

In [ ]:
process_images('openai')

**Package versions**

In [25]:
from importlib.metadata import version
from IPython.display import Markdown, display # pip install ipython
import sys

packages = ['anthropic', 'google-genai', 'mistralai', 'openai', 'openpyxl', 'pandas', 'pillow', 'python-dotenv']

text = f"Python version: {sys.version}\n\n"
for i in packages:
    text += f"[{i}](https://pypi.org/project/{i}/) version: {version(i)}\n\n"
display(Markdown(text))

Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]

[anthropic](https://pypi.org/project/anthropic/) version: 0.75.0

[google-genai](https://pypi.org/project/google-genai/) version: 1.55.0

[mistralai](https://pypi.org/project/mistralai/) version: 1.9.11

[openai](https://pypi.org/project/openai/) version: 2.2.0

[openpyxl](https://pypi.org/project/openpyxl/) version: 3.1.5

[pandas](https://pypi.org/project/pandas/) version: 2.3.2

[pillow](https://pypi.org/project/pillow/) version: 11.3.0

[python-dotenv](https://pypi.org/project/python-dotenv/) version: 1.1.1

